# SpongeBob Character Classifier — YOLOv8-cls Training Pipeline

This notebook trains a YOLOv8 **classification** model to recognize 6 characters from SpongeBob SquarePants:

`SpongeBob, Patrick, Squidward, Sandy, Plankton, Mr. Krabs`

**How to use this notebook**
1. Runtime → Change runtime type → GPU (T4 is enough).
2. Run the cells top to bottom.
3. Section 2 (data collection) downloads images automatically from the web. If you already have your own images, skip Section 2 and just drop your images into the folder structure described in Section 3.
4. Section 4 trains the model. The final weights are saved as `model.pt`.

> Note: this notebook downloads publicly available images from the web for a personal/educational classroom exercise. Review the downloaded images before using them for anything beyond coursework, and respect the source sites' terms of use and Nickelodeon's IP rights.


## 1. Setup — install dependencies

In [ ]:
!pip install -q ultralytics icrawler pillow imagehash tqdm
import os, shutil, random
from pathlib import Path

CLASSES = ["spongebob", "patrick", "squidward", "sandy", "plankton", "mr_krabs"]
SEARCH_TERMS = {
    "spongebob": ["SpongeBob SquarePants character screenshot"],
    "patrick":   ["Patrick Star SpongeBob character screenshot"],
    "squidward": ["Squidward Tentacles SpongeBob character screenshot"],
    "sandy":     ["Sandy Cheeks SpongeBob character screenshot"],
    "plankton":  ["Plankton SpongeBob character screenshot"],
    "mr_krabs":  ["Mr Krabs SpongeBob character screenshot"],
}

RAW_DIR = Path("data_raw")
RAW_DIR.mkdir(exist_ok=True)
print("Classes:", CLASSES)


## 2. Dataset collection

We use `icrawler` to pull candidate images per class from a search engine, then manually/automatically clean them.
Aim for **150–300 raw images per class** before cleaning (cleaning will remove ~20–40%), so the final
dataset has at least **100 usable images per class** (600+ total). More is better — try 300–500/class if you have time.

You can re-run this cell with different `max_num` values or add extra search terms per class to widen the pool.


In [ ]:
from icrawler.builtin import BingImageCrawler

MAX_PER_TERM = 200  # increase for a bigger dataset

for cls, terms in SEARCH_TERMS.items():
    out_dir = RAW_DIR / cls
    out_dir.mkdir(parents=True, exist_ok=True)
    for term in terms:
        crawler = BingImageCrawler(storage={"root_dir": str(out_dir)})
        crawler.crawl(keyword=term, max_num=MAX_PER_TERM, min_size=(150, 150))
    print(cls, "->", len(list(out_dir.glob("*"))), "images downloaded")


## 3. Dataset cleaning & annotation

For a **classification** task, "annotation" = making sure every image sits in the correct class folder and
is a clean, single-character, non-duplicate image. Steps performed below:

1. **Corrupt-file removal** — drop files PIL can't open.
2. **Deduplication** — perceptual hashing (`imagehash`) removes near-identical/duplicate frames.
3. **Manual review (recommended)** — after the automated pass, open `data_clean/<class>/` in the Colab
   file browser (or download the folder) and manually delete: wrong character, multiple characters in
   frame, fan art that doesn't resemble the official character design, logos/merchandise-only images,
   and heavily edited/meme images. This manual pass is the actual "labeling QA" step for this project —
   log how many images you removed and why in your report.
4. **Train / val / test split** — 70% / 20% / 10% per class, into the folder layout YOLO classification expects:

```
dataset/
  train/<class_name>/*.jpg
  val/<class_name>/*.jpg
  test/<class_name>/*.jpg
```


In [ ]:
from PIL import Image
import imagehash

CLEAN_DIR = Path("data_clean")
if CLEAN_DIR.exists():
    shutil.rmtree(CLEAN_DIR)

removed_corrupt = 0
removed_dupe = 0

for cls in CLASSES:
    src = RAW_DIR / cls
    dst = CLEAN_DIR / cls
    dst.mkdir(parents=True, exist_ok=True)
    seen_hashes = set()
    for f in sorted(src.glob("*")):
        try:
            img = Image.open(f).convert("RGB")
        except Exception:
            removed_corrupt += 1
            continue
        h = str(imagehash.phash(img))
        if h in seen_hashes:
            removed_dupe += 1
            continue
        seen_hashes.add(h)
        img.save(dst / f"{f.stem}.jpg", "JPEG", quality=95)
    print(cls, "-> kept", len(list(dst.glob('*'))))

print(f"Removed {removed_corrupt} corrupt files and {removed_dupe} duplicates in total.")
print("\n>>> STOP HERE and manually spot-check data_clean/<class>/ folders before continuing. <<<")


In [ ]:
import random

SPLIT_DIR = Path("dataset")
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

random.seed(42)
TRAIN, VAL, TEST = 0.7, 0.2, 0.1

for cls in CLASSES:
    files = list((CLEAN_DIR / cls).glob("*.jpg"))
    random.shuffle(files)
    n = len(files)
    n_train = int(n * TRAIN)
    n_val = int(n * VAL)
    split_map = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    for split, split_files in split_map.items():
        out = SPLIT_DIR / split / cls
        out.mkdir(parents=True, exist_ok=True)
        for f in split_files:
            shutil.copy(f, out / f.name)
    print(f"{cls}: train={n_train}, val={n_val}, test={n - n_train - n_val}")


## 4. Train the YOLOv8 classification model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")  # nano classification backbone, fast to train

results = model.train(
    data=str(SPLIT_DIR.resolve()),
    epochs=50,
    imgsz=224,
    batch=32,
    patience=10,
    project="runs_spongebob",
    name="cls_v1",
)


## 5. Evaluate on the held-out test split

In [ ]:
metrics = model.val(data=str(SPLIT_DIR.resolve()), split="test")
print(metrics.results_dict)


## 6. Export the trained weights as `model.pt`

In [ ]:
best_weights = Path("runs_spongebob/cls_v1/weights/best.pt")
shutil.copy(best_weights, "model.pt")
print("Saved model.pt ->", Path("model.pt").resolve())

# Optional: download to your machine
from google.colab import files as colab_files
colab_files.download("model.pt")


## 7. Quick inference sanity check

In [ ]:
trained = YOLO("model.pt")
sample = next((SPLIT_DIR / "test").rglob("*.jpg"))
pred = trained(sample)
print("True label folder:", sample.parent.name)
print("Predicted:", pred[0].names[pred[0].probs.top1], "confidence:", float(pred[0].probs.top1conf))
